<a href="https://colab.research.google.com/github/TheFaridKarami/farid-ai-lab/blob/main/projects/nlp/wnut17-robust-ner/notebooks/wnut17_multi_model_ner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
! pip install -q seqeval

In [2]:
import sys
import numpy as np
import torch, time
from torch import nn
import transformers
from transformers import (AutoTokenizer,
                          AutoModelForTokenClassification,
                          DataCollatorForTokenClassification,
                          TrainingArguments,
                          Trainer,
                          set_seed)

import datasets
from seqeval.metrics import classification_report, accuracy_score
from seqeval.scheme import IOB2

In [3]:
wnut = datasets.load_dataset("flaitenberger/wnut_17")

Repo card metadata block was not found. Setting CardData to empty.


In [4]:
label_list = [label for label in wnut["train"].features["ner_tags"].feature.names]
id2label =   {i:label for i, label in enumerate(label_list)}
label2id =   {label:i for i, label in id2label.items()}


In [5]:
shown = 0
for ex in wnut["train"]:

  if any(t != 0 for t in ex["ner_tags"]):
    print(" ".join(ex["tokens"]))
    print([(tok, id2label[t]) for tok, t in zip(ex["tokens"], ex["ner_tags"])])
    print()
  shown += 1

  if shown == 5:
    break

@paulwalk It 's the view from where I 'm living for two weeks . Empire State Building = ESB . Pretty bad storm here last evening .
[('@paulwalk', 'O'), ('It', 'O'), ("'s", 'O'), ('the', 'O'), ('view', 'O'), ('from', 'O'), ('where', 'O'), ('I', 'O'), ("'m", 'O'), ('living', 'O'), ('for', 'O'), ('two', 'O'), ('weeks', 'O'), ('.', 'O'), ('Empire', 'B-location'), ('State', 'I-location'), ('Building', 'I-location'), ('=', 'O'), ('ESB', 'B-location'), ('.', 'O'), ('Pretty', 'O'), ('bad', 'O'), ('storm', 'O'), ('here', 'O'), ('last', 'O'), ('evening', 'O'), ('.', 'O')]

From Green Newsfeed : AHFA extends deadline for Sage Award to Nov . 5 http://tinyurl.com/24agj38
[('From', 'O'), ('Green', 'O'), ('Newsfeed', 'O'), (':', 'O'), ('AHFA', 'B-group'), ('extends', 'O'), ('deadline', 'O'), ('for', 'O'), ('Sage', 'O'), ('Award', 'O'), ('to', 'O'), ('Nov', 'O'), ('.', 'O'), ('5', 'O'), ('http://tinyurl.com/24agj38', 'O')]

Pxleyes Top 50 Photography Contest Pictures of August 2010 ... http://bit.ly/b

In [6]:
for split in wnut:
  lens = [len(ex["tokens"]) for ex in wnut[split]]
  print(split, "mean:",round(np.mean(lens),2), "max: ", np.max(lens), "median: ", np.median(lens), "95th pct: ", np.quantile(lens, 0.95))

train mean: 18.48 max:  41 median:  18.0 95th pct:  30.0
validation mean: 15.59 max:  82 median:  13.0 95th pct:  38.0
test mean: 18.18 max:  105 median:  15.0 95th pct:  42.0


In [7]:
from collections import Counter

for split in wnut:
  c = Counter(id2label[t] for ex in wnut[split] for t in ex["ner_tags"])
  ents = {l[2:]: n for l , n in c.items() if l.startswith("B-")}
  total = sum(c.values())
  print(split, "tokens:", total, "O share:", round(c["O"] / total, 3))
  print(ents, "total entities:", sum(ents.values()))

train tokens: 62730 O share: 0.95
{'location': 548, 'group': 264, 'corporation': 221, 'person': 660, 'creative-work': 140, 'product': 142} total entities: 1975
validation tokens: 15733 O share: 0.921
{'location': 74, 'creative-work': 105, 'person': 470, 'product': 114, 'group': 39, 'corporation': 34} total entities: 836
test tokens: 23394 O share: 0.926
{'location': 150, 'group': 165, 'person': 429, 'creative-work': 142, 'corporation': 66, 'product': 127} total entities: 1079


In [8]:
tok = AutoTokenizer.from_pretrained("bert-base-cased")

words = ["Pxleyes", "Top", "50", "Photography", "Contest", "#photography"]

enc = tok(words, is_split_into_words= True)

print(tok.convert_ids_to_tokens(enc["input_ids"]))
print(enc.word_ids())

['[CLS]', 'P', '##xley', '##es', 'Top', '50', 'Photography', 'Contest', '#', 'photography', '[SEP]']
[None, 0, 0, 0, 1, 2, 3, 4, 5, 5, None]


In [9]:
def tokenize_and_align(batch):

  enc = tok(batch["tokens"], is_split_into_words=True, truncation=True, max_length=256)
  all_labels = []
  for i, tags in enumerate(batch["ner_tags"]):
    prev = None
    labs = []
    for w in enc.word_ids(batch_index=i):
      if w is None:
        labs.append(-100)
      elif w != prev:
        labs.append(tags[w])
      else:
        labs.append(-100)
      prev = w

    all_labels.append(labs)
  enc["labels"] = all_labels

  return enc

In [10]:
tokenized = wnut.map(tokenize_and_align, batched = True, remove_columns = wnut["train"].column_names)

for split in tokenized:
  lost = sum(len(ex["tokens"]) - sum(l != -100 for l in lab) for ex, lab in zip(wnut[split], tokenized[split]["labels"]))
  print(split, "max_tokens:", max(len(x) for x in tokenized[split]["input_ids"]), "words lost:", lost)

train max_tokens: 90 words lost: 0
validation max_tokens: 84 words lost: 0
test max_tokens: 198 words lost: 0


In [11]:
val_gold = [[id2label[t] for t in ex["ner_tags"]] for ex in wnut["validation"]]
test_gold = [[id2label[t] for t in ex["ner_tags"]] for ex in wnut["test"]]

In [12]:
model = AutoModelForTokenClassification.from_pretrained("bert-base-cased", num_labels=len(label_list), id2label=id2label, label2id=label2id)
print(model.classifier)
print(sum(p.numel() for p in model.parameters()), sum(p.numel() for p in model.parameters() if p.requires_grad))

collator = DataCollatorForTokenClassification(tok)
batch = collator([tokenized["train"][i] for i in range(3)])
print({k: v.shape for k, v in batch.items()})
print(batch["labels"][0])

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly i

Linear(in_features=768, out_features=13, bias=True)
107729677 107729677
{'input_ids': torch.Size([3, 35]), 'token_type_ids': torch.Size([3, 35]), 'attention_mask': torch.Size([3, 35]), 'labels': torch.Size([3, 35])}
tensor([-100,    0, -100, -100, -100,    0,    0, -100,    0,    0,    0,    0,
           0,    0, -100,    0,    0,    0,    0,    0,    7,    8,    8,    0,
           7, -100,    0,    0,    0,    0,    0,    0,    0,    0, -100])


In [13]:
def evaluate_ner(gold, pred):

    rep = classification_report(
        gold,
        pred,
        mode="strict",
        scheme=IOB2,
        output_dict=True,
        zero_division=0)

    out = {"precision": rep["micro avg"]["precision"],
           "recall": rep["micro avg"]["recall"],
           "f1": rep["micro avg"]["f1-score"],
           "token_acc": accuracy_score(gold, pred)}

    out["per_type_f1"] = {k: v["f1-score"] for k, v in rep.items() if "avg" not in k}

    return out

In [21]:
def to_tags(logits, labels):

  preds = np.argmax(logits, axis = -1)
  gold, pred = [], []
  for pr, lb in zip(preds, labels):
    gold.append([id2label[l] for l in lb if l != -100])
    pred.append([id2label[q] for q, l in zip(pr, lb) if l != -100])

  return gold, pred

def compute_metrics(p):
  gold, pred = to_tags(*p)
  r = evaluate_ner(gold, pred)
  return {
      "precision":r["precision"],
      "f1":r["f1"],
      "recall": r["recall"],
      "accuracy": r["token_acc"]}




In [19]:
b = collator([tokenized["validation"][i] for i in range(64)])
labels = b["labels"].numpy()
logits = np.eye(len(label_list))[np.where(labels == -100, 0, labels)]
print(compute_metrics((logits, labels)))

{'precision': np.float64(1.0), 'f1-score': np.float64(1.0), 'recall': np.float64(1.0), 'accuracy': 1.0}


In [22]:
import torch, time
from transformers import Trainer, TrainingArguments, set_seed

set_seed(42)

model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-cased",
    num_labels = len(label_list),
    id2label = id2label,
    label2id = label2id)

args = TrainingArguments(
    output_dir="bert_base_cased_ckpt",
    learning_rate = 5e-5,
    per_device_train_batch_size = 32,
    per_device_eval_batch_size = 64,
    num_train_epochs = 10,
    weight_decay = 0.01,
    warmup_steps = 107,
    fp16  =True,
    eval_strategy = "epoch",
    save_strategy = "epoch",
    save_total_limit = 1,
    load_best_model_at_end = True,
    metric_for_best_model = "f1",
    seed = 42,
    report_to = "none")


trainer = Trainer(model = model,
                  args = args,
                  train_dataset = tokenized["train"],
                  eval_dataset  = tokenized["validation"],
                  data_collator = collator,
                  processing_class = tok,
                  compute_metrics = compute_metrics)

torch.cuda.reset_peak_memory_stats()
t0 = time.time()

trainer.train()


train_time = time.time() - t0
peak_gb = torch.cuda.max_memory_allocated() / 1e9
print("train time (s):", round(train_time), "peak GPU memory (GB):", round(peak_gb, 2))

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly i

Epoch,Training Loss,Validation Loss,Precision,F1,Recall,Accuracy
1,No log,0.243535,0.640232,0.489283,0.395933,0.944194
2,No log,0.215693,0.676692,0.526316,0.430622,0.949024
3,No log,0.232288,0.672161,0.531114,0.438995,0.951058
4,No log,0.229611,0.639517,0.565710,0.507177,0.952838
5,0.185357,0.265698,0.630603,0.550236,0.488038,0.952012
6,0.185357,0.275898,0.665625,0.577236,0.509569,0.953791
7,0.185357,0.313522,0.652648,0.566982,0.501196,0.953029
8,0.185357,0.351152,0.673504,0.554539,0.471292,0.952647
9,0.185357,0.337716,0.665079,0.571623,0.501196,0.953474
10,0.004516,0.352923,0.655172,0.552249,0.477273,0.952457


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (5 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

train time (s): 395 peak GPU memory (GB): 4.06


In [23]:
out = trainer.predict(tokenized["validation"])
gold, pred = to_tags(out.predictions, out.label_ids)
print(evaluate_ner(gold, pred))
print(gold == val_gold)
print(trainer.state.best_metric, trainer.state.best_model_checkpoint)

{'precision': np.float64(0.665625), 'recall': np.float64(0.5095693779904307), 'f1': np.float64(0.5772357723577236), 'token_acc': 0.953854954554122, 'per_type_f1': {'corporation': np.float64(0.1724137931034483), 'creative-work': np.float64(0.2916666666666667), 'group': np.float64(0.16216216216216214), 'location': np.float64(0.6119402985074627), 'person': np.float64(0.7258064516129032), 'product': np.float64(0.3838383838383838)}}
True
0.5772357723577236 bert_base_cased_ckpt/checkpoint-642
